In [3]:
import collections
import sys
import os

def solve(N: int, M: int, K: int, S: int, T: int,
          vertex_capacities: list[int],
          edges: list[tuple[int, int, int, int]]) -> int:
    """
    Executes the Latency-Constrained Dark Pool Liquidity Routing algorithm.
    Utilizes a Time-Expanded Graph with Vertex Splitting and Edmonds-Karp Max-Flow.
    """
    INF = int(1e15)
    TOTAL_NODES = N * (K + 1) * 2 + 2
    SUPER_SOURCE = TOTAL_NODES - 2
    SUPER_SINK = TOTAL_NODES - 1
    
    def in_node(v: int, k: int) -> int:
        return v * 2 * (K + 1) + 2 * k
        
    def out_node(v: int, k: int) -> int:
        return v * 2 * (K + 1) + 2 * k + 1

    graph = [collections.defaultdict(int) for _ in range(TOTAL_NODES)]
    
    def add_edge(u: int, v: int, capacity: int) -> None:
        if capacity > 0:
            graph[u][v] += capacity
            if v not in graph[u]: 
                pass 
            graph[v][u] += 0 

    # A. Vertex Splitting (Processing Capacities)
    for v in range(N):
        cap = INF if vertex_capacities[v] == -1 else vertex_capacities[v]
        for k in range(K + 1):
            add_edge(in_node(v, k), out_node(v, k), cap)
            
    # B. Resting Liquidity (Wait Edges)
    for v in range(N):
        for k in range(K):
            add_edge(out_node(v, k), in_node(v, k + 1), INF)
            
    # C. Spatial-Temporal Connections
    for u, v, cap, tau in edges:
        for k in range(K + 1 - tau):
            add_edge(out_node(u, k), in_node(v, k + tau), cap)
            
    # D. Super Nodes Routing
    add_edge(SUPER_SOURCE, in_node(S, 0), INF)
    for k in range(K + 1):
        add_edge(out_node(T, k), SUPER_SINK, INF)

    # Edmonds-Karp Execution
    max_flow = 0
    while True:
        parent = {SUPER_SOURCE: -1}
        queue = collections.deque([(SUPER_SOURCE, INF)])
        path_flow = 0
        
        while queue:
            curr, flow = queue.popleft()
            if curr == SUPER_SINK:
                path_flow = flow
                break
                
            for nxt, capacity in graph[curr].items():
                if nxt not in parent and capacity > 0:
                    parent[nxt] = curr
                    queue.append((nxt, min(flow, capacity)))
                    
        if path_flow == 0:
            break
            
        max_flow += path_flow
        curr = SUPER_SINK
        while curr != SUPER_SOURCE:
            prev = parent[curr]
            graph[prev][curr] -= path_flow
            graph[curr][prev] += path_flow
            curr = prev
            
    return max_flow

def main():
    """
    User interface function to handle I/O and execute the graph algorithm.
    """
    print("="*60)
    print(" LATENCY-CONSTRAINED DARK POOL LIQUIDITY ROUTING ENGINE")
    print("="*60)
    
    filepath = input("\nEnter the path to the network text file: ").strip()
    
    if not os.path.exists(filepath):
        print(f"\n[Error] The file '{filepath}' could not be located.")
        sys.exit(1)
        
    try:
        # Prompt for the systemic constraints
        print("\n--- System Constraints ---")
        N = int(input("Total number of physical broker nodes (N): "))
        K = int(input("Strict latency bound in microseconds (K): "))
        S = int(input("Origin Node ID (Source): "))
        T = int(input("Market Node ID (Sink): "))
        
        # We allow a universal vertex capacity for simplicity via CLI, 
        # but the architecture supports passing a list of distinct capacities.
        def_cap_input = input("Default node processing capacity (-1 for infinite): ")
        default_vc = int(def_cap_input)
        vertex_capacities = [default_vc] * N
        # --- MODIFICATION: Uncap the Source and Sink ---
        vertex_capacities[S] = -1  # Removes the bottleneck on the originating institution
        vertex_capacities[T] = -1  # Removes the bottleneck on the final market exchange
        # -----------------------------------------------
        
    except ValueError:
        print("\n[Error] System constraints must be provided as valid integers.")
        sys.exit(1)

    edges = []
    
    # Parse the file
    with open(filepath, 'r') as file:
        for line_idx, line in enumerate(file):
            clean_line = line.strip()
            
            # Skip empty lines and comments
            if not clean_line or clean_line.startswith('#'):
                continue
                
            parts = clean_line.split()
            if len(parts) >= 3:
                try:
                    u = int(parts[0])
                    v = int(parts[1])
                    w = int(parts[2])  # w -> Edge Capacity
                    
                    # If the user included latency in the file, we capture it.
                    # Otherwise, assume the standard 1 microsecond hop.
                    tau = int(parts[3]) if len(parts) >= 4 else 1
                    
                    edges.append((u, v, w, tau))
                except ValueError:
                    print(f"[Warning] Skipping malformed line {line_idx + 1}: '{clean_line}'")
            else:
                print(f"[Warning] Line {line_idx + 1} does not contain 'u v w'. Skipping.")

    print("\n" + "-"*60)
    print(f"Loaded {len(edges)} connections. Constructing Time-Expanded Graph...")
    
    # Execute the solver
    max_liquidity = solve(N, len(edges), K, S, T, vertex_capacities, edges)
    
    # Output the result to the console
    print("-"*60)
    print("=== EXECUTION RESULTS ===")
    print(f"Maximum routeable liquidity within {K} microseconds: {max_liquidity} shares")
    print("="*60)

if __name__ == "__main__":
    main()

 LATENCY-CONSTRAINED DARK POOL LIQUIDITY ROUTING ENGINE



Enter the path to the network text file:  network.txt



--- System Constraints ---


Total number of physical broker nodes (N):  6
Strict latency bound in microseconds (K):  5
Origin Node ID (Source):  0
Market Node ID (Sink):  5
Default node processing capacity (-1 for infinite):  40



------------------------------------------------------------
Loaded 8 connections. Constructing Time-Expanded Graph...
------------------------------------------------------------
=== EXECUTION RESULTS ===
Maximum routeable liquidity within 5 microseconds: 105 shares
